# 14 — Channel-stacked versus spatiotemporal FNO

The model in notebooks `06`–`07` treats the 14-day history as input **channels**.
This notebook compares it against an operator that treats time as an explicit
**dimension**.

`FNO2d` is unchanged and remains the baseline. `FNO3d` lives in a separate module.

> **Guardrail.** A 3-D operator is more structurally elaborate. That is not
> evidence that it is better. The conclusion must come from the fixed
> out-of-sample comparison below, and if the elaborate model loses, that is the
> result.

## What actually changes

The channel-stacked spectral layer learns one complex matrix per retained
**spatial** frequency:

\[
\hat{v}_{out}(k_y, k_x) = R_\theta(k_y, k_x)\,\hat{v}_{in}(k_y, k_x),
\qquad R_\theta(k_y,k_x)\in\mathbb{C}^{C_{out}\times C_{in}}.
\]

Because the lookback *is* the channel axis, \(R_\theta\) mixes days with a dense,
unstructured matrix. **Permuting the input days permutes channels** — nothing in
the architecture encodes that day \(t-1\) neighbours day \(t-2\). The model can
learn that ordering from data, but it is not given it.

Transforming time explicitly replaces that with multiplication in temporal
frequency:

\[
\hat{v}_{out}(k_t, k_y, k_x) = R_\theta(k_t, k_y, k_x)\,\hat{v}_{in}(k_t, k_y, k_x),
\]

which by the convolution theorem **is a convolution along time**. Three
consequences:

1. **Translation equivariance in time**, by construction rather than by training.
   A shifted history produces a correspondingly shifted response. This is asserted
   in `tests/test_model3d.py`.
2. **Truncation becomes a temporal smoothness prior.** Keeping `modes_t` temporal
   frequencies discards fast day-to-day variation, just as spatial truncation
   discards small scales. That is a modelling assumption, not a free improvement.
3. **Parameter scaling changes**, derived and then measured below.

One more thing the temporal FFT assumes: that the time window is **periodic**. A
14-day history is not, exactly as the regional spatial domain is not — so the same
padding argument applies along time.

In [ ]:
from pathlib import Path

import numpy as np
import torch

from oisst_fno.data import SSTSequenceDataset, Standardizer, open_oisst, temporal_split
from oisst_fno.experiment import measure_cost, set_global_seed, width_for_parameter_budget
from oisst_fno.metrics import (
    daily_rmse,
    parameter_count,
    rmse,
    skill_score,
    spectral_error_energy_by_band,
    temporal_increment_correlation,
    temporal_variability_ratio,
)
from oisst_fno.model import FNO2d
from oisst_fno.model3d import FNO3d

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_global_seed(42)

HEIGHT, WIDTH = 81, 101   # Northeast Atlantic study grid
LOOKBACK = 14
HORIZON = 14              # forecast the next 14 days; read leads 1, 7, 14 from one output
LEADS = (1, 7, 14)
DEPTH = 4
print("device:", DEVICE)

## Capacity: the cost of representing time

A 2-D spectral layer holds \(2\,m_y m_x w^2\) weights — two bands because the real
FFT halves only the last axis. The 3-D layer holds \(4\,m_t m_y m_x w^2\): four
corner bands, since time and latitude each need both signs.

So at equal width the ratio is \((4 m_t)/2\). The measurement below should confirm
that derivation rather than merely accompany it.

In [ ]:
baseline = FNO2d(
    in_channels=LOOKBACK + 1,   # 14 SST days + ocean mask
    out_channels=HORIZON,       # the whole trajectory as output channels
    width=48,
    modes_y=16,
    modes_x=16,
    depth=DEPTH,
)
target_parameters = parameter_count(baseline)

equal_width = FNO3d(steps_out=HORIZON, width=48, modes_t=4, modes_y=16, modes_x=16, depth=DEPTH)

print(f"2-D channel-stacked (width 48): {target_parameters:>12,} parameters")
print(f"3-D at the same width 48      : {parameter_count(equal_width):>12,} parameters")
print(f"ratio                         : {parameter_count(equal_width) / target_parameters:>12.2f}x")
print()
print("Predicted ratio from the scaling relation:")
print("  2-D spectral weights: 2 bands  x modes_y x modes_x x width^2")
print("  3-D spectral weights: 4 bands  x modes_t x modes_y x modes_x x width^2")
print(f"  -> (4 x modes_t) / 2 = {(4 * 4) / 2:.0f}x at modes_t = 4")

### Matching the budget

Comparing architectures is only informative with capacity held roughly fixed;
otherwise "the elaborate model won" and "the bigger model won" are
indistinguishable. The residual mismatch is reported rather than glossed over.

In [ ]:
match = width_for_parameter_budget(
    lambda w: FNO3d(steps_out=HORIZON, width=w, modes_t=4, modes_y=16, modes_x=16, depth=DEPTH),
    target_parameters,
    max_width=64,
)
matched = FNO3d(
    steps_out=HORIZON, width=match.width, modes_t=4, modes_y=16, modes_x=16, depth=DEPTH
)

print(f"matched 3-D width      : {match.width}")
print(f"matched 3-D parameters : {match.parameters:,} ({match.relative_error:+.1%} vs target)")
print()
print("The 3-D model therefore runs at roughly a third of the 2-D latent width for the")
print("same parameter budget. That narrower latent space is part of the trade, not an")
print("accident of configuration.")

### Computational cost

Equal parameters does **not** mean equal cost. Three FFT axes and a time dimension
in every activation change both time and memory, and a model that is several times
more expensive per forecast has to earn that back in skill.

In [ ]:
sample_2d = torch.randn(2, LOOKBACK + 1, HEIGHT, WIDTH, device=DEVICE)
sample_3d = torch.randn(2, 1, LOOKBACK, HEIGHT, WIDTH, device=DEVICE)

print(f"{'model':>20} {'parameters':>12} {'fwd s':>9} {'fwd+bwd s':>11} {'peak MiB':>10}")
costs = {}
for name, model, sample in (
    ("2-D channel-stack", baseline.to(DEVICE), sample_2d),
    (f"3-D matched (w{match.width})", matched.to(DEVICE), sample_3d),
    ("3-D equal width 48", equal_width.to(DEVICE), sample_3d),
):
    forward = measure_cost(model, sample, repeats=3, warmup=1)
    both = measure_cost(model, sample, repeats=2, warmup=1, backward=True)
    costs[name] = (forward, both)
    memory = f"{forward.peak_gpu_mb:.0f}" if forward.peak_gpu_mb is not None else "n/a (cpu)"
    print(
        f"{name:>20} {forward.parameters:>12,} {forward.seconds_per_pass:>9.4f} "
        f"{both.seconds_per_pass:>11.4f} {memory:>10}"
    )

## The comparison

Both arms consume identical windows and predict identical targets: 14 days of
history to the following 14 days, with leads 1, 7, and 14 read from one forecast.
Normalisation is fitted on the training split only.

In [ ]:
# Both architectures must consume identical windows and predict identical targets.
DATA_PATH = sorted((ROOT / "data" / "raw").glob("oisst_*_ne_atlantic.nc"))[-1]
TRAIN_END, VALIDATION_END = "2024-12-31", "2025-12-31"

sst = open_oisst(DATA_PATH)["sst"]
train_da, val_da, test_da = temporal_split(sst, TRAIN_END, VALIDATION_END)
scaler = Standardizer.fit(train_da.values)   # training split only

splits = {
    "train": SSTSequenceDataset(scaler.transform(train_da.values), LOOKBACK, HORIZON),
    "val": SSTSequenceDataset(scaler.transform(val_da.values), LOOKBACK, HORIZON),
    "test": SSTSequenceDataset(scaler.transform(test_da.values), LOOKBACK, HORIZON),
}
for name, dataset in splits.items():
    print(f"{name:>6}: {len(dataset)} windows, target leads {dataset.target_offsets()[[0, 6, 13]]}")

x, y, mask = splits["test"][0]
print(f"\nwindow: history {tuple(x.shape)}, target {tuple(y.shape)}, mask {tuple(mask.shape)}")
print("2-D reads history as channels; 3-D reads the same tensor with time as a dimension.")

### Training

Each arm is a full training run under the matched budget. **No skill numbers are
reported until this has been executed** — the cell below raises rather than
emitting placeholders.

In [ ]:
# Training both arms under a matched budget is the expensive step and has not been run.
# Same seed, epochs, patience, optimizer, and schedule as notebook 07, per arm.
raise NotImplementedError(
    "Train both arms with notebook 07's loop under the matched budget above, then "
    "continue. No skill numbers are reported until this has actually been run."
)

In [ ]:
# Evaluation recipe, applied identically to both arms once predictions exist.
#
# Predictions have shape [window, lead, lat, lon] in degrees Celsius after
# scaler.inverse_transform, on identical target dates for both models.
#
# 1. Pointwise error and persistence skill, per lead time:
#
#      for lead in LEADS:
#          model_rmse = rmse(pred[:, lead - 1], truth[:, lead - 1], mask)
#          persist_rmse = rmse(persistence[:, lead - 1], truth[:, lead - 1], mask)
#          skill = skill_score(model_rmse, persist_rmse)
#
#    Persistence stays the primary null model at every lead. A model that beats the other
#    architecture but not persistence has not earned a positive result.
#
# 2. Paired daily differences with block-bootstrap intervals, respecting serial
#    dependence, exactly as in notebook 08.
#
# 3. Temporal coherence, which single-lead evaluation cannot see:
#
#      temporal_increment_correlation(pred[i], truth[i], mask)   # does it move like truth?
#      temporal_variability_ratio(pred[i], truth[i], mask)       # or is it frozen?
#
#    A ratio well below 1 means the forecast evolves more slowly than reality. Squared
#    error rewards exactly that, so low RMSE with a low ratio is a finding to report, not
#    a success to celebrate.
#
# 4. Spatial spectra per lead via spectral_error_energy_by_band, using the same mask and
#    taper for both models so the comparison is relative scale attribution.
print("Evaluation recipe defined; run after training.")

## How to read the result

The four evidence axes answer different questions, and they can disagree:

| Axis | Question |
|---|---|
| Pointwise error, persistence skill | Is the forecast close to the truth, and closer than doing nothing? |
| Temporal coherence | Does the forecast *move* like the truth, or is it frozen? |
| Spatial spectra | Which scales carry the error reduction? |
| Time and memory | What does the improvement cost? |

A plausible outcome is that the 3-D operator improves temporal coherence — its
temporal convolution structure is built for that — while not improving RMSE at
lead 7. That would be a real and reportable finding, not a failure.

### Conclusions that are not permitted

- **"3-D is better because it is principled."** Structural elaboration is not
  evidence. The prompt's guardrail is explicit and the fixed out-of-sample
  comparison decides.
- **"3-D wins" from a run at unequal capacity.** Match the budget or state the
  mismatch.
- **"3-D is worse" from an undertrained run.** Check the learning-curve verdict
  from notebook `07` first; an `underfit` verdict invalidates the comparison in
  either direction.
- **Either architecture "works" while losing to persistence.** Persistence remains
  the primary null model at every lead.